In [ ]:
import asyncio
import hashlib
import inspect
import json
import os
import resource
import sys
import threading
import time
import uuid
from pathlib import Path

import numpy as np
from IPython.display import display

import xyg
from xyg import _native
from xyg.graph_layout import GraphLayoutController

ROOT = Path(xyg.__file__).resolve().parents[2]
assert Path(os.environ["XYG_GRAPH_SOURCE_ROOT"]).resolve() == ROOT, str(ROOT)
sys.path.insert(0, str(ROOT))
from scripts.bench_cose_progressive import ladder_edges  # noqa: E402 - repo-only fixture

N = int(os.environ.get("XYG_NOTEBOOK_SCALE", "1000"))
assert N in (1000, 10000, 100000)


def ids(tag, count):
    out = np.zeros((count, 16), dtype=np.uint8)
    out[:, :6] = np.frombuffer((1760000000000).to_bytes(6, "big"), dtype=np.uint8)
    out[:, 6] = 0x70 | tag
    out[:, 8] = 0x80
    out[:, 9:] = np.arange(count, dtype=">u8").view(np.uint8).reshape(count, 8)[:, 1:]
    return out


def compose(count):
    nodes = ids(1, count)
    sources, targets = ladder_edges(count)
    return xyg.compose_graphforge(
        base={
            "node_uuid": nodes,
            "edge_uuid": ids(2, len(sources)),
            "edge_source_uuid": nodes[sources],
            "edge_target_uuid": nodes[targets],
            "generation": str(
                uuid.UUID(bytes=bytes([0, 0, 0, 0, 0, 0, 0x73, 0, 0x80]) + count.to_bytes(7, "big"))
            ),
            "directed": False,
        }
    )


setup_started_ns = time.monotonic_ns()
composition = compose(N)
assert composition.nodes.count == N and composition.edges.count == 2 * N
other_composition = composition if N == 1000 else compose(1000)
clicks, hovers, selections, other_clicks = [], [], [], []
chart = xyg.graphforge_chart(
    composition,
    title=f"Native notebook {N}",
    on_click=clicks.append,
    on_hover=hovers.append,
    on_select=lambda value: selections.append(
        {
            "rows": value.rows(),
            "graphforge": getattr(value, "graphforge", []),
            "error": getattr(value, "graphforge_error", None),
        }
    ),
)
widget = chart.widget()
_client_sha = hashlib.sha256(widget._esm.encode()).hexdigest()
assert _client_sha == hashlib.sha256((ROOT / "python/xyg/static/index.js").read_bytes()).hexdigest()
other = xyg.graphforge_chart(
    other_composition, title="Independent 1k notebook peer", on_click=other_clicks.append
).widget()
assert chart._figure is None
_layout_defaults = inspect.signature(GraphLayoutController.layout).parameters
setup_finished_ns = time.monotonic_ns()
display(widget)
display(other)
print(
    "XYG scale setup",
    json.dumps(
        {
            "pythonRoot": str(ROOT),
            "kernelExecutable": sys.executable,
            "kernelVersion": sys.version,
            "clientSha256": _client_sha,
            "primaryInstance": widget.spec["instance"],
            "peerInstance": other.spec["instance"],
            "nodes": N,
            "edges": 2 * N,
            "peerNodes": 1000,
            "peerEdges": 2000,
            "compositionBytes": len(composition.bytes),
            "compositionSha256": hashlib.sha256(composition.bytes).hexdigest(),
            "nativeSha256": hashlib.sha256(Path(_native._lib._name).read_bytes()).hexdigest(),
            "setupMs": (setup_finished_ns - setup_started_ns) / 1e6,
            "layout": chart._graphforge_mark.props["layout"],
            "iterations": chart._graphforge_mark.props["iterations"],
            "chunkSteps": _layout_defaults["chunk_steps"].default,
            "maxWallMs": _layout_defaults["max_wall_ms"].default,
        }
    ),
)

In [ ]:
print(
    "XYG scale responsive",
    json.dumps(
        {
            "timeNs": time.monotonic_ns(),
            "primaryLive": widget._task is not None and not widget._task.done(),
            "peerLive": other._task is not None and not other._task.done(),
            "primaryRevision": widget._revision,
            "peerRevision": other._revision,
            "failed": widget._failed,
            "nodes": N,
            "edges": 2 * N,
        }
    ),
)

In [ ]:
def snapshot(owner):
    figure = owner._figure
    identity = getattr(figure, "_notebook_graph_identity", None) if figure is not None else None
    return {
        "ack": owner._ack.is_set() if owner._ack else None,
        "building": owner._building,
        "pendingApi": len(owner._pending_api),
        "selection": owner.view_state().get("selection"),
        "step": figure._graph_meta[0]["iterations"] if figure else None,
        "revision": owner._revision,
        "failed": owner._failed,
        "running": owner._task is not None and not owner._task.done(),
        "figure": figure is not None,
        "identityError": int(identity._error)
        if identity is not None and identity._error is not None
        else None,
        "identityPrepared": identity is not None and identity.session is not None,
        "sourcePositions": len(identity._positions[0]) if identity is not None else None,
        "sourceNodes": identity.composition.nodes.count if identity is not None else None,
        "sourceEdges": identity.composition.edges.count if identity is not None else None,
    }


await asyncio.sleep(0.05)
print(
    "XYG scale report",
    json.dumps(
        {
            "primary": snapshot(widget),
            "peer": snapshot(other),
            "clicks": clicks,
            "hovers": hovers,
            "selections": selections,
            "otherClicks": other_clicks,
            "expected": composition.identify("node", 0),
            "rssLinuxKiB": resource.getrusage(resource.RUSAGE_SELF).ru_maxrss,
            "workerThreads": [t.name for t in threading.enumerate() if t.name.startswith("xyg-")],
        }
    ),
)

In [ ]:
widget.select(uuids=[composition.nodes.uuid[0]])
print("XYG scale selected UUID", composition.nodes.uuid[0])

In [ ]:
def snapshot(owner):
    figure = owner._figure
    identity = getattr(figure, "_notebook_graph_identity", None) if figure is not None else None
    return {
        "ack": owner._ack.is_set() if owner._ack else None,
        "building": owner._building,
        "pendingApi": len(owner._pending_api),
        "selection": owner.view_state().get("selection"),
        "step": figure._graph_meta[0]["iterations"] if figure else None,
        "revision": owner._revision,
        "failed": owner._failed,
        "running": owner._task is not None and not owner._task.done(),
        "figure": figure is not None,
        "identityError": int(identity._error)
        if identity is not None and identity._error is not None
        else None,
        "identityPrepared": identity is not None and identity.session is not None,
        "sourcePositions": len(identity._positions[0]) if identity is not None else None,
        "sourceNodes": identity.composition.nodes.count if identity is not None else None,
        "sourceEdges": identity.composition.edges.count if identity is not None else None,
    }


await asyncio.sleep(0.05)
print(
    "XYG scale report",
    json.dumps(
        {
            "primary": snapshot(widget),
            "peer": snapshot(other),
            "clicks": clicks,
            "hovers": hovers,
            "selections": selections,
            "otherClicks": other_clicks,
            "expected": composition.identify("node", 0),
            "rssLinuxKiB": resource.getrusage(resource.RUSAGE_SELF).ru_maxrss,
            "workerThreads": [t.name for t in threading.enumerate() if t.name.startswith("xyg-")],
        }
    ),
)

In [ ]:
owners = [
    f._notebook_graph_identity.session
    for w in (widget, other)
    for f in w._figures.values()
    if getattr(f, "_notebook_graph_identity", None) is not None
    and f._notebook_graph_identity.session is not None
]
workers = [t for t in threading.enumerate() if t.name.startswith(("xyg-layout", "xyg-graph-build"))]
widget.close()
other.close()
for _ in range(1000):
    if all(not t.is_alive() for t in workers):
        break
    await asyncio.sleep(0.01)
assert all(not t.is_alive() for t in workers), "native/build worker did not retire"
assert all(o._handle is None for o in owners), "native identity owner did not close"
print(
    "XYG scale cleanup",
    json.dumps(
        {
            "threadsStopped": True,
            "identityOwnersClosed": True,
            "owners": len(owners),
            "workers": len(workers),
        }
    ),
)